# Video Studio — `sdxl` on Colab (manual backup)

Kaggle is the studio's automated GPU provider. Use this notebook only when Kaggle has no GPU free.

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Stay on the page.
2. When it finishes, the browser downloads **`vs-result-sdxl.zip`**. Give that file to Claude
   (upload it to the session or commit it to the branch); Claude runs `vs-colab import`.
3. The last cell releases the GPU so your Colab allowance is not wasted.

Colab rules: run it yourself in this notebook UI; no SSH, remote desktop or web UIs; one account;
no deepfakes — lip sync only on faces you own, AI-generated people, or people who agreed in writing.
Free Colab limits are not published and change; a run can be stopped if you are idle.

Same code as the Kaggle job (packed by `vs-colab build` from `gpu-jobs/`); models are downloaded
from their official Hugging Face / GitHub sources at pinned revisions.

In [ ]:
# 1. GPU check + map the Kaggle paths the job uses onto /content
import os, subprocess, pathlib
r = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
assert r.returncode == 0, 'No GPU: Runtime > Change runtime type > T4 GPU, then Run all again'
print(r.stdout)
for d in ('/content/kaggle/working', '/content/kaggle/temp'):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
if not os.path.exists('/kaggle'):
    os.symlink('/content/kaggle', '/kaggle')


In [ ]:
# 3. the studio job (generated — do not edit; rebuild with vs-colab build)
JOB = '# --- packed by vs-gpu: inputs, params, shared runtime (gpu-jobs/_common.py), then the job ---\nimport base64, json, os, pathlib, time\n_T0 = time.time()\nIN = pathlib.Path("/kaggle/working/in"); OUT = pathlib.Path("/kaggle/working/out")\nIN.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)\nPARAMS = json.loads(\'{"model": "sdxl", "items": [{"id": "kid-reading", "prompt": "portrait of a cheerful 7-year-old Egyptian girl reading a colourful picture book on a cosy sofa, face clearly visible, looking at camera, warm afternoon window light, storybook illustration style", "width": 720, "height": 1280, "seed": 11}, {"id": "courier", "prompt": "a young courier on an orange scooter on a sunny Cairo street at sunrise, cinematic photo, 35mm, shallow depth of field", "width": 1280, "height": 720, "seed": 7}, {"id": "presenter", "prompt": "photo of a friendly young Arab woman news presenter facing the camera, head and shoulders, studio lighting, neutral background, sharp focus", "width": 1024, "height": 1024, "seed": 21}], "item_budget_s": 240, "max_job_s": 1500, "_need_gpu": true, "_need_internet": true, "parallel": false}\')\nfor _n, _b in json.loads(\'{}\').items():\n    (IN / _n).write_bytes(base64.b64decode(_b))\n_COMMON_B64 = \'IyBTaGFyZWQgcnVudGltZSBmb3IgZXZlcnkgc3R1ZGlvIEdQVSBqb2IgKHBhY2tlZCBieSB2cy1ncHU7IHJ1bnMgaW5zaWRlIHRoZSBLYWdnbGUga2VybmVsIGFuZCBpdHMKIyBwZXItR1BVIGNoaWxkIHByb2Nlc3NlcykuIFByb3ZpZGVzOiByZXBvcnQoKSwgZ3B1X2luZm8oKSwgaXNfdDQoKSwgcGlja19kdHlwZSgpLCBzaGFyZCgpLCBsb2FkX3BpcGUoKS4KIyBLYWdnbGUgZnJlZSB0aWVyID0gMnggVGVzbGEgVDQgKHNtNzUsIDE0LjYgR0IgZWFjaCk6IGZhc3QgZnAxNiB0ZW5zb3IgY29yZXMsIGJmMTYgb25seSBlbXVsYXRlZC4KaW1wb3J0IGpzb24sIG9zLCBwYXRobGliLCBzdWJwcm9jZXNzLCBzeXMsIHRpbWUKClNIQVJEID0gaW50KG9zLmVudmlyb24uZ2V0KCJWU19TSEFSRCIsICItMSIpKSAgICAgICAgICAjID49IDAgaW5zaWRlIGEgcGVyLUdQVSBjaGlsZCBwcm9jZXNzCgoKZGVmIHJlcG9ydCgqKmt3KToKICAgIHAgPSBPVVQgLyAoInJlcG9ydC5qc29uIiBpZiBTSEFSRCA8IDAgZWxzZSBmInJlcG9ydC5zaGFyZHtTSEFSRH0uanNvbiIpCiAgICBkID0ganNvbi5sb2FkcyhwLnJlYWRfdGV4dCgpKSBpZiBwLmV4aXN0cygpIGVsc2Uge30KICAgIGQudXBkYXRlKGt3KTsgZFsiZWxhcHNlZF9zIl0gPSByb3VuZCh0aW1lLnRpbWUoKSAtIF9UMCwgMSk7IHAud3JpdGVfdGV4dChqc29uLmR1bXBzKGQsIGluZGVudD0yKSkKCgpkZWYgX3ZyYW1fcGVhaygpOiAgICAgICAgICAgIyBiZW5jaG1hcms6IHBlYWsgVlJBTSBwZXIgdmlzaWJsZSBHUFUsIHdyaXR0ZW4gYXQgZXhpdCBvZiBldmVyeSAoY2hpbGQpIHByb2Nlc3MKICAgIHRyeToKICAgICAgICBpbXBvcnQgdG9yY2gKICAgICAgICBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgICAgICByZXBvcnQoKip7InZyYW1fcGVha19nYiI6IFtyb3VuZCh0b3JjaC5jdWRhLm1heF9tZW1vcnlfYWxsb2NhdGVkKGkpIC8gMioqMzAsIDIpCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZvciBpIGluIHJhbmdlKHRvcmNoLmN1ZGEuZGV2aWNlX2NvdW50KCkpXX0pCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHBhc3MKCgpkZWYgX3dhdGNoZG9nKGxpbWl0X3MpOiAgICAjIGhhcmQgam9iIGJ1ZGdldDogbmV2ZXIgbGV0IGEgc3R1Y2sgam9iIGJ1cm4gR1BVIHF1b3RhIChLYWdnbGUgYWxsb3dzIDEyIGgpCiAgICBpbXBvcnQgdGhyZWFkaW5nCiAgICBkZWYgZmlyZSgpOgogICAgICAgIHJlcG9ydChlcnJvcj1mImpvYiBleGNlZWRlZCBpdHMgYnVkZ2V0IG9mIHtsaW1pdF9zfSBzIGFuZCB3YXMgc3RvcHBlZCAocmFpc2UgbWF4X2pvYl9zIG9ubHkgaWYgbmVlZGVkKSIsCiAgICAgICAgICAgICAgIHRpbWVkX291dD1UcnVlKQogICAgICAgIF92cmFtX3BlYWsoKTsgb3MuX2V4aXQoNSkKICAgIHQgPSB0aHJlYWRpbmcuVGltZXIobGltaXRfcywgZmlyZSk7IHQuZGFlbW9uID0gVHJ1ZTsgdC5zdGFydCgpCgoKaW1wb3J0IGF0ZXhpdAphdGV4aXQucmVnaXN0ZXIoX3ZyYW1fcGVhaykKaWYgU0hBUkQgPCAwIGFuZCBub3Qgb3MuZW52aXJvbi5nZXQoIlZTX0ZBS0VfR1BVUyIpOgogICAgX3dhdGNoZG9nKGludChQQVJBTVMuZ2V0KCJtYXhfam9iX3MiLCAzNjAwKSkpCgoKZGVmIHN0ZXBfZ3VhcmQodG90YWxfc3RlcHMsIGJ1ZGdldF9zLCBsYWJlbD0iaXRlbSIpOgogICAgIiIiZGlmZnVzZXJzIGNhbGxiYWNrX29uX3N0ZXBfZW5kOiBzdG9wIG9uIE5hTi9pbmYgbGF0ZW50cyBhbmQgd2hlbiB0aGUgcHJvamVjdGVkIHRpbWUgb2YgdGhpcyBpdGVtIGV4Y2VlZHMKICAgIGJ1ZGdldF9zIChlYXJseSBmYWlsdXJlIGRldGVjdGlvbiwgbWVhc3VyZWQgcGVyIHN0ZXApLiBSYWlzZXMgRmxvYXRpbmdQb2ludEVycm9yIC8gVGltZW91dEVycm9yLiIiIgogICAgdDAgPSB0aW1lLnRpbWUoKQogICAgc3RhdHMgPSB7InN0ZXBzIjogMCwgImRlbm9pc2VfcyI6IDAuMH0KICAgIGRlZiBjYihwaXBlLCBzdGVwLCB0cywga3cpOgogICAgICAgIGltcG9ydCB0b3JjaAogICAgICAgIGxhdCA9IGt3LmdldCgibGF0ZW50cyIpCiAgICAgICAgaWYgbGF0IGlzIG5vdCBOb25lIGFuZCBub3QgdG9yY2guaXNmaW5pdGUobGF0KS5hbGwoKToKICAgICAgICAgICAgcmFpc2UgRmxvYXRpbmdQb2ludEVycm9yKGYie2xhYmVsfTogbm9uLWZpbml0ZSBsYXRlbnRzIGF0IHN0ZXAge3N0ZXB9IikKICAgICAgICBlbCA9IHRpbWUudGltZSgpIC0gdDAKICAgICAgICBpZiBzdGVwID49IDEgYW5kIGVsIC8gKHN0ZXAgKyAxKSAqIHRvdGFsX3N0ZXBzID4gYnVkZ2V0X3M6CiAgICAgICAgICAgIHJhaXNlIFRpbWVvdXRFcnJvcihmIntsYWJlbH06IHtlbCAvIChzdGVwICsgMSk6LjFmfSBzL3N0ZXAgeCB7dG90YWxfc3RlcHN9IHN0ZXBzID4gYnVkZ2V0IHtidWRnZXRfc30gcyIpCiAgICAgICAgc3RhdHMudXBkYXRlKHN0ZXBzPXN0ZXAgKyAxLCBkZW5vaXNlX3M9cm91bmQoZWwsIDEpKQogICAgICAgIHJldHVybiBrdwogICAgY2Iuc3RhdHMgPSBzdGF0cwogICAgcmV0dXJuIGNiCgoKZGVmIGdwdV9pbmZvKCk6CiAgICBpZiBvcy5lbnZpcm9uLmdldCgiVlNfRkFLRV9HUFVTIik6ICAgICAgICMgQ1BVIHNlbGYtdGVzdCBvZiBwYWNraW5nIGFuZCBmYW4tb3V0ICh0ZXN0cy90ZXN0X2dwdV9ydW50aW1lLnNoKQogICAgICAgIHJldHVybiB7Im4iOiBpbnQob3MuZW52aXJvblsiVlNfRkFLRV9HUFVTIl0pLCAibmFtZSI6ICJmYWtlIiwgInZyYW1fZ2IiOiAxNC42LCAiY2MiOiA3LjV9CiAgICBpbXBvcnQgdG9yY2gKICAgIG4gPSB0b3JjaC5jdWRhLmRldmljZV9jb3VudCgpCiAgICBpZiBub3QgbjoKICAgICAgICByZXR1cm4geyJuIjogMH0KICAgIHAgPSB0b3JjaC5jdWRhLmdldF9kZXZpY2VfcHJvcGVydGllcygwKQogICAgcmV0dXJuIHsibiI6IG4sICJuYW1lIjogcC5uYW1lLCAidnJhbV9nYiI6IHJvdW5kKHAudG90YWxfbWVtb3J5IC8gMioqMzAsIDEpLCAiY2MiOiBwLm1ham9yICsgcC5taW5vciAvIDEwfQoKCmRlZiBpc190NCgpOgogICAgZyA9IGdwdV9pbmZvKCkKICAgIHJldHVybiBnLmdldCgibiIsIDApID4gMCBhbmQgZ1siY2MiXSA8IDguMAoKCmRlZiBwaWNrX2R0eXBlKGZwMTZfc2FmZSk6CiAgICAiIiJmcDE2IG9uIFQ0IGZvciBtb2RlbHMga25vd24gdG8gYmUgZnAxNi1zYWZlICg0LTh4IGZhc3RlciB0aGFuIGVtdWxhdGVkIGJmMTYpOyBiZjE2IG90aGVyd2lzZS4KICAgIFBBUkFNU1siZHR5cGUiXSA9IGZwMTYgfCBiZjE2IHwgZnAzMiBvdmVycmlkZXMuIiIiCiAgICBpbXBvcnQgdG9yY2gKICAgIHdhbnQgPSBQQVJBTVMuZ2V0KCJkdHlwZSIsICJhdXRvIikKICAgIGlmIHdhbnQgaW4gKCJmcDE2IiwgImJmMTYiLCAiZnAzMiIpOgogICAgICAgIHJldHVybiB7ImZwMTYiOiB0b3JjaC5mbG9hdDE2LCAiYmYxNiI6IHRvcmNoLmJmbG9hdDE2LCAiZnAzMiI6IHRvcmNoLmZsb2F0MzJ9W3dhbnRdCiAgICBpZiBpc190NCgpOgogICAgICAgIHJldHVybiB0b3JjaC5mbG9hdDE2IGlmIGZwMTZfc2FmZSBlbHNlIHRvcmNoLmJmbG9hdDE2CiAgICByZXR1cm4gdG9yY2guYmZsb2F0MTYKCgpjbGFzcyBfRG9uZShFeGNlcHRpb24pOgogICAgcGFzcwoKCmRlZiBzaGFyZChpdGVtcyk6CiAgICAiIiJTbWFsbCBtb2RlbHM6IG9uZSBwcm9jZXNzIHBlciBHUFUsIGl0ZW1zIHNwbGl0IHJvdW5kLXJvYmluICgyeCB0aHJvdWdocHV0IG9uIDJ4IFQ0KS4KICAgIFJldHVybnMgdGhpcyBwcm9jZXNzJ3MgaXRlbXM7IGluIHRoZSBwYXJlbnQgaXQgd2FpdHMgZm9yIHRoZSBjaGlsZHJlbiwgbWVyZ2VzIHJlcG9ydHMgYW5kIHN0b3BzIHRoZSBqb2IuIiIiCiAgICBuID0gZ3B1X2luZm8oKS5nZXQoIm4iLCAwKQogICAgaWYgU0hBUkQgPj0gMDoKICAgICAgICByZXR1cm4gaXRlbXNbU0hBUkQ6OmludChvcy5lbnZpcm9uWyJWU19OU0hBUkQiXSldCiAgICBpZiBuIDwgMiBvciBsZW4oaXRlbXMpIDwgMiBvciBub3QgUEFSQU1TLmdldCgicGFyYWxsZWwiLCBUcnVlKToKICAgICAgICByZXR1cm4gaXRlbXMKICAgIHRtcCA9IHBhdGhsaWIuUGF0aCgiL2thZ2dsZS90ZW1wIik7IHRtcC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAodG1wIC8gInZzX3BhcmFtcy5qc29uIikud3JpdGVfdGV4dChqc29uLmR1bXBzKFBBUkFNUykpCiAgICAodG1wIC8gInZzX2NoaWxkLnB5Iikud3JpdGVfdGV4dCgKICAgICAgICAiaW1wb3J0IGJhc2U2NCwganNvbiwgcGF0aGxpYiwgdGltZVxuX1QwID0gdGltZS50aW1lKClcbiIKICAgICAgICAiSU4gPSBwYXRobGliLlBhdGgoJy9rYWdnbGUvd29ya2luZy9pbicpOyBPVVQgPSBwYXRobGliLlBhdGgoJy9rYWdnbGUvd29ya2luZy9vdXQnKVxuIgogICAgICAgICJQQVJBTVMgPSBqc29uLmxvYWQob3BlbignL2thZ2dsZS90ZW1wL3ZzX3BhcmFtcy5qc29uJykpXG4iCiAgICAgICAgZiJfQ09NTU9OX0I2NCA9IHtfQ09NTU9OX0I2NCFyfVxuX0pPQl9CNjQgPSB7X0pPQl9CNjQhcn1cbiIKICAgICAgICAiZXhlYyhiYXNlNjQuYjY0ZGVjb2RlKF9DT01NT05fQjY0KS5kZWNvZGUoKSlcbiIKICAgICAgICAiZXhlYyhjb21waWxlKGJhc2U2NC5iNjRkZWNvZGUoX0pPQl9CNjQpLmRlY29kZSgpLCAnam9iLnB5JywgJ2V4ZWMnKSlcbiIpCiAgICBwcm9jcyA9IFtzdWJwcm9jZXNzLlBvcGVuKFtzeXMuZXhlY3V0YWJsZSwgc3RyKHRtcCAvICJ2c19jaGlsZC5weSIpXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZW52PXsqKm9zLmVudmlyb24sICJWU19TSEFSRCI6IHN0cihpKSwgIlZTX05TSEFSRCI6IHN0cihuKSwgIkNVREFfVklTSUJMRV9ERVZJQ0VTIjogc3RyKGkpfSkKICAgICAgICAgICAgIGZvciBpIGluIHJhbmdlKG4pXQogICAgY29kZXMgPSBbcC53YWl0KCkgZm9yIHAgaW4gcHJvY3NdCiAgICBzaGFyZHMgPSBbXQogICAgZm9yIGkgaW4gcmFuZ2Uobik6CiAgICAgICAgZiA9IE9VVCAvIGYicmVwb3J0LnNoYXJke2l9Lmpzb24iCiAgICAgICAgaWYgZi5leGlzdHMoKToKICAgICAgICAgICAgc2hhcmRzLmFwcGVuZChqc29uLmxvYWRzKGYucmVhZF90ZXh0KCkpKTsgZi51bmxpbmsoKQogICAgcmVwb3J0KHBhcmFsbGVsX2dwdXM9biwgc2hhcmRzPXNoYXJkcywgc2hhcmRfZXhpdF9jb2Rlcz1jb2RlcykKICAgIGlmIGNvZGVzIGFuZCBhbGwoY29kZXMpOgogICAgICAgIHJlcG9ydChlcnJvcj1mImV2ZXJ5IEdQVSB3b3JrZXIgZmFpbGVkOiBleGl0IGNvZGVzIHtjb2Rlc30iKQogICAgZWxpZiBhbnkoY29kZXMpOiAgICAgICAgICAgICAgICAgICAgICAgICAgIyBwYXJ0aWFsIHJlc3VsdDoga2VlcCB3aGF0IHdhcyBtYWRlLCB2cy1wcm9kdWNlIGZhbGxzIGJhY2sgZm9yIHRoZSByZXN0CiAgICAgICAgcmVwb3J0KHdhcm5pbmc9ZiJhIEdQVSB3b3JrZXIgZmFpbGVkOiBleGl0IGNvZGVzIHtjb2Rlc30iKQogICAgcmFpc2UgX0RvbmUoKQoKCmRlZiBsb2FkX3BpcGUoY2xzLCByZXBvLCByZXZpc2lvbiwgZHR5cGUsIGZwOF9vaz1UcnVlLCBwcmVmZXI9Tm9uZSwgKiprdyk6CiAgICAiIiJCaWcgZGlmZnVzaW9uIG1vZGVscyBvbiAyeCBUNCwgYmVzdCBzdHJhdGVneSBmaXJzdDoKICAgICAgIDEpIGRldmljZV9tYXA9J2JhbGFuY2VkJyDigJQgdGV4dCBlbmNvZGVyIG9uIG9uZSBUNCwgdHJhbnNmb3JtZXIgKyBWQUUgb24gdGhlIG90aGVyIChubyBDUFUgcm91bmQtdHJpcHMpCiAgICAgICAyKSBmcDggd2VpZ2h0IHN0b3JhZ2UgZm9yIHRoZSB0cmFuc2Zvcm1lciArIG1vZGVsIENQVSBvZmZsb2FkIG9uIG9uZSBUNAogICAgICAgMykgc2VxdWVudGlhbCBDUFUgb2ZmbG9hZCAoc2xvd2VzdCwgYWx3YXlzIGZpdHMpCiAgICBQQVJBTVNbInBsYWNlbWVudCJdID0gYmFsYW5jZWQgfCBvZmZsb2FkIHwgc2VxdWVudGlhbCBmb3JjZXMgb25lLiIiIgogICAgaW1wb3J0IHRvcmNoCiAgICBmb3JjZSA9IHByZWZlciBvciBQQVJBTVMuZ2V0KCJwbGFjZW1lbnQiKQogICAgZXJyb3JzID0gW10KICAgIGlmIChmb3JjZSBpbiAoTm9uZSwgImJhbGFuY2VkIikpIGFuZCBncHVfaW5mbygpLmdldCgibiIsIDApID49IDI6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBwaXBlID0gY2xzLmZyb21fcHJldHJhaW5lZChyZXBvLCByZXZpc2lvbj1yZXZpc2lvbiwgdG9yY2hfZHR5cGU9ZHR5cGUsIGRldmljZV9tYXA9ImJhbGFuY2VkIiwgKiprdykKICAgICAgICAgICAgZnA4ID0gZnA4X29rIGFuZCBncHVfaW5mbygpWyJ2cmFtX2diIl0gPCAyMCBhbmQgaGFzYXR0cihwaXBlLCAidHJhbnNmb3JtZXIiKQogICAgICAgICAgICBpZiBmcDg6ICAgIyBtZWFzdXJlZDogWi1JbWFnZSdzIDZCIHRyYW5zZm9ybWVyIGFsb25lICgxMiBHQiBiZjE2KSArIGF0dGVudGlvbiBvdmVyZmxvd3MgYSAxNC42IEdCIFQ0CiAgICAgICAgICAgICAgICBwaXBlLnRyYW5zZm9ybWVyLmVuYWJsZV9sYXllcndpc2VfY2FzdGluZyhzdG9yYWdlX2R0eXBlPXRvcmNoLmZsb2F0OF9lNG0zZm4sIGNvbXB1dGVfZHR5cGU9ZHR5cGUpCiAgICAgICAgICAgICAgICB0b3JjaC5jdWRhLmVtcHR5X2NhY2hlKCkKICAgICAgICAgICAgcmVwb3J0KHBsYWNlbWVudD0iYmFsYW5jZWQgKDIgR1BVcykiICsgKCIgKyBmcDggdHJhbnNmb3JtZXIgd2VpZ2h0cyIgaWYgZnA4IGVsc2UgIiIpKQogICAgICAgICAgICByZXR1cm4gcGlwZQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgZXJyb3JzLmFwcGVuZChmImJhbGFuY2VkOiB7dHlwZShlKS5fX25hbWVfX306IHtzdHIoZSlbOjIwMF19IikKICAgIGlmIGZvcmNlIGluIChOb25lLCAib2ZmbG9hZCIpOgogICAgICAgIHRyeToKICAgICAgICAgICAgcGlwZSA9IGNscy5mcm9tX3ByZXRyYWluZWQocmVwbywgcmV2aXNpb249cmV2aXNpb24sIHRvcmNoX2R0eXBlPWR0eXBlLCAqKmt3KQogICAgICAgICAgICBpZiBmcDhfb2sgYW5kIGdwdV9pbmZvKClbInZyYW1fZ2IiXSA8IDIwIGFuZCBoYXNhdHRyKHBpcGUsICJ0cmFuc2Zvcm1lciIpOgogICAgICAgICAgICAgICAgcGlwZS50cmFuc2Zvcm1lci5lbmFibGVfbGF5ZXJ3aXNlX2Nhc3Rpbmcoc3RvcmFnZV9kdHlwZT10b3JjaC5mbG9hdDhfZTRtM2ZuLCBjb21wdXRlX2R0eXBlPWR0eXBlKQogICAgICAgICAgICBwaXBlLmVuYWJsZV9tb2RlbF9jcHVfb2ZmbG9hZChncHVfaWQ9MCkKICAgICAgICAgICAgcmVwb3J0KHBsYWNlbWVudD0iZnA4IHRyYW5zZm9ybWVyICsgbW9kZWwgQ1BVIG9mZmxvYWQgKDEgR1BVKSIsIHBsYWNlbWVudF9lcnJvcnM9ZXJyb3JzKQogICAgICAgICAgICByZXR1cm4gcGlwZQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgZXJyb3JzLmFwcGVuZChmIm9mZmxvYWQ6IHt0eXBlKGUpLl9fbmFtZV9ffToge3N0cihlKVs6MjAwXX0iKQogICAgcGlwZSA9IGNscy5mcm9tX3ByZXRyYWluZWQocmVwbywgcmV2aXNpb249cmV2aXNpb24sIHRvcmNoX2R0eXBlPWR0eXBlLCAqKmt3KQogICAgcGlwZS5lbmFibGVfc2VxdWVudGlhbF9jcHVfb2ZmbG9hZCgpCiAgICByZXBvcnQocGxhY2VtZW50PSJzZXF1ZW50aWFsIENQVSBvZmZsb2FkIiwgcGxhY2VtZW50X2Vycm9ycz1lcnJvcnMpCiAgICByZXR1cm4gcGlwZQoKCmRlZiB2YWVfdGlsaW5nKHBpcGUpOgogICAgZm9yIG5hbWUgaW4gKCJ2YWUiLCk6CiAgICAgICAgbSA9IGdldGF0dHIocGlwZSwgbmFtZSwgTm9uZSkKICAgICAgICBmb3IgZm4gaW4gKCJlbmFibGVfdGlsaW5nIiwgImVuYWJsZV9zbGljaW5nIik6CiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIGdldGF0dHIobSwgZm4pKCkKICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgICAgIHBhc3MKCgpkZWYgbG9va3NfYnJva2VuKGFycik6CiAgICAiIiJOYU4vYmxhY2sgb3V0cHV0IGNoZWNrIChmcDE2IG92ZXJmbG93IHNob3dzIHVwIGFzIGJsYWNrIGZyYW1lcykuIiIiCiAgICBpbXBvcnQgbnVtcHkgYXMgbnAKICAgIGEgPSBucC5hc2FycmF5KGFyciwgZHR5cGU9bnAuZmxvYXQzMikKICAgIHJldHVybiAobm90IG5wLmlzZmluaXRlKGEpLmFsbCgpKSBvciBmbG9hdChhLm1lYW4oKSkgPCAyLjAK\'\n_JOB_B64 = \'IyBHUFUgam9iOiB0ZXh0LXRvLWltYWdlLCB0dW5lZCBmb3IgS2FnZ2xlJ3MgMnggVGVzbGEgVDQgKDE0LjYgR0IgZWFjaCwgc203NSkuCiMgUEFSQU1TOiB7Im1vZGVsIjogInNkeGwiIHwgInppbWFnZSIsICJpdGVtcyI6IFt7ImlkIiwicHJvbXB0IiwibmVnYXRpdmUiLCJ3aWR0aCIsImhlaWdodCIsInNlZWQifV0sICJzdGVwcyI6IE4sCiMgICAgICAgICAgIml0ZW1fYnVkZ2V0X3MiOiAyNDAsICJtYXhfam9iX3MiOiAzNjAwfQojICAgc2R4bCAgICBERUZBVUxUIG9uIFQ0LiBTRFhMIGJhc2UgMS4wIChPcGVuUkFJTCsrKSArIGZwMTYtZml4IFZBRSAoTUlUKTogbmF0aXZlIGZwMTYsIH43IEdCLCBvbmUgY29weSBwZXIgVDQgc28KIyAgICAgICAgICAgYm90aCBHUFVzIHdvcmsgaW4gcGFyYWxsZWwsIDI4IHN0ZXBzLiBSZWxpYWJsZSBvbiB0aGUgZnJlZSB0aWVyLgojICAgemltYWdlICBBRFZBTkNFRCAob3B0aW9uYWwpLiBaLUltYWdlLVR1cmJvIDZCIChBcGFjaGUtMi4wKTogMTIgR0IgdHJhbnNmb3JtZXIg4oCUIG9uIFQ0IGl0IG5lZWRzIGJvdGggR1BVcyArCiMgICAgICAgICAgIGZwOCB3ZWlnaHRzIChPT00gaW4gYmYxNiwgbWVhc3VyZWQpLiBGYWxscyBiYWNrIHRvIFNEWEwgYXV0b21hdGljYWxseSBvbiBhbnkgZmFpbHVyZS4KIyBFdmVyeSBpbWFnZSBpcyBjaGVja2VkIGZvciBOYU4vYmxhY2sgb3V0cHV0IChmcDE2IG92ZXJmbG93KSBhbmQgcmV0cmllZCBvbmNlIGluIGEgc2FmZXIgbW9kZS4KaW1wb3J0IG9zLCBzdWJwcm9jZXNzLCBzeXMKb3MuZW52aXJvbi5zZXRkZWZhdWx0KCJQWVRPUkNIX0NVREFfQUxMT0NfQ09ORiIsICJleHBhbmRhYmxlX3NlZ21lbnRzOlRydWUiKQpzdWJwcm9jZXNzLnJ1bihbc3lzLmV4ZWN1dGFibGUsICItbSIsICJwaXAiLCAiaW5zdGFsbCIsICItcSIsICJkaWZmdXNlcnM9PTAuNDMuMCIsICJhY2NlbGVyYXRlIiwgInRyYW5zZm9ybWVycyIsCiAgICAgICAgICAgICAgICAic2FmZXRlbnNvcnMiXSwgY2hlY2s9VHJ1ZSkKaW1wb3J0IHRvcmNoCk1PREVMUyA9IHsiemltYWdlIjogKCJUb25neWktTUFJL1otSW1hZ2UtVHVyYm8iLCAiZjMzMjA3MmFhNzhiIiksCiAgICAgICAgICAic2R4bCI6ICgic3RhYmlsaXR5YWkvc3RhYmxlLWRpZmZ1c2lvbi14bC1iYXNlLTEuMCIsICI0NjIxNjU5ODQwMzAiKX0KbW9kZWwgPSBQQVJBTVMuZ2V0KCJtb2RlbCIpIG9yICgic2R4bCIgaWYgaXNfdDQoKSBlbHNlICJ6aW1hZ2UiKQoKCmRlZiBnZW5femltYWdlKGl0ZW1zKToKICAgIGZyb20gZGlmZnVzZXJzIGltcG9ydCBaSW1hZ2VQaXBlbGluZQogICAgdCA9IHRpbWUudGltZSgpCiAgICBkdHlwZSA9IHBpY2tfZHR5cGUoZnAxNl9zYWZlPUZhbHNlKSAgICAgICAgICAjIFotSW1hZ2UgaXMgdHJhaW5lZCBpbiBiZjE2OyBmcDE2IG9ubHkgaWYgUEFSQU1TIGR0eXBlPWZwMTYKICAgIHBpcGUgPSBsb2FkX3BpcGUoWkltYWdlUGlwZWxpbmUsICpNT0RFTFNbInppbWFnZSJdLCBkdHlwZSkKICAgIHZhZV90aWxpbmcocGlwZSkKICAgIHJlcG9ydChsb2FkX3M9cm91bmQodGltZS50aW1lKCkgLSB0LCAxKSwgZHR5cGU9c3RyKGR0eXBlKSwgbW9kZWw9InotaW1hZ2UtdHVyYm8iKQogICAgZG9uZSA9IFtdCiAgICBmb3IgaXQgaW4gaXRlbXM6CiAgICAgICAgcyA9IHRpbWUudGltZSgpCiAgICAgICAgZyA9IHRvcmNoLkdlbmVyYXRvcigiY3B1IikubWFudWFsX3NlZWQoaXQuZ2V0KCJzZWVkIiwgMCkpCiAgICAgICAgaW1nID0gcGlwZShwcm9tcHQ9aXRbInByb21wdCJdLCBoZWlnaHQ9aXQuZ2V0KCJoZWlnaHQiLCA3MjApLCB3aWR0aD1pdC5nZXQoIndpZHRoIiwgMTI4MCksCiAgICAgICAgICAgICAgICAgICBudW1faW5mZXJlbmNlX3N0ZXBzPVBBUkFNUy5nZXQoInN0ZXBzIiwgOCksIGd1aWRhbmNlX3NjYWxlPTAuMCwgZ2VuZXJhdG9yPWcpLmltYWdlc1swXQogICAgICAgIGlmIGxvb2tzX2Jyb2tlbihpbWcpOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ7aXRbJ2lkJ119OiBibGFjay9OYU4gb3V0cHV0IikKICAgICAgICBpbWcuc2F2ZShPVVQgLyBmIntpdFsnaWQnXX0ucG5nIik7IGRvbmUuYXBwZW5kKHsiaWQiOiBpdFsiaWQiXSwgInMiOiByb3VuZCh0aW1lLnRpbWUoKSAtIHMsIDEpfSkKICAgICAgICByZXBvcnQoaW1hZ2VzPWRvbmUpCgoKZGVmIGdlbl9zZHhsKGl0ZW1zKToKICAgIGZyb20gZGlmZnVzZXJzIGltcG9ydCBTdGFibGVEaWZmdXNpb25YTFBpcGVsaW5lLCBBdXRvZW5jb2RlcktMCiAgICBpdGVtcyA9IHNoYXJkKGl0ZW1zKSAgICAgICAgICAgICAgICAgICAgICAgICAjIG9uZSBTRFhMIGNvcHkgcGVyIFQ0CiAgICB0ID0gdGltZS50aW1lKCkKICAgIHZhZSA9IEF1dG9lbmNvZGVyS0wuZnJvbV9wcmV0cmFpbmVkKCJtYWRlYnlvbGxpbi9zZHhsLXZhZS1mcDE2LWZpeCIsIHJldmlzaW9uPSIyMDdiMTE2ZGFlNzAiLCB0b3JjaF9kdHlwZT10b3JjaC5mbG9hdDE2KQogICAgcGlwZSA9IFN0YWJsZURpZmZ1c2lvblhMUGlwZWxpbmUuZnJvbV9wcmV0cmFpbmVkKE1PREVMU1sic2R4bCJdWzBdLCByZXZpc2lvbj1NT0RFTFNbInNkeGwiXVsxXSwgdmFlPXZhZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB0b3JjaF9kdHlwZT10b3JjaC5mbG9hdDE2LCB2YXJpYW50PSJmcDE2IikudG8oImN1ZGEiKQogICAgdmFlX3RpbGluZyhwaXBlKSAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyBkaWZmdXNlcnMgMC40MzogdGlsaW5nIGxpdmVzIG9uIHRoZSBWQUUKICAgIHJlcG9ydChsb2FkX3M9cm91bmQodGltZS50aW1lKCkgLSB0LCAxKSwgZHR5cGU9InRvcmNoLmZsb2F0MTYiLCBtb2RlbD0ic2R4bC1iYXNlLTEuMCIpCiAgICBkb25lID0gW10KICAgIG5lZyA9ICJibHVycnksIGxvd3JlcywgZGVmb3JtZWQsIGV4dHJhIGZpbmdlcnMsIHdhdGVybWFyaywgdGV4dCwgbG9nbyIKICAgIHN0ZXBzID0gUEFSQU1TLmdldCgic3RlcHNfc2R4bCIsIDI4KQogICAgZG9uZSwgZmFpbGVkID0gW10sIFtdCiAgICBmb3IgaXQgaW4gaXRlbXM6CiAgICAgICAgdywgaCA9IGl0LmdldCgid2lkdGgiLCAxMzQ0KSwgaXQuZ2V0KCJoZWlnaHQiLCA3NjgpCiAgICAgICAgdywgaCA9ICgxMzQ0LCA3NjgpIGlmIHcgPiBoIGVsc2UgKCg3NjgsIDEzNDQpIGlmIGggPiB3IGVsc2UgKDEwMjQsIDEwMjQpKSAgICMgU0RYTC1uYXRpdmUgYnVja2V0cwogICAgICAgIGZvciBhdHRlbXB0IGluIHJhbmdlKDIpOiAgICAgICAgICAgICAgICAgIyBvbmUgcmV0cnkgd2l0aCBhbm90aGVyIHNlZWQgb24gTmFOL2JsYWNrIG91dHB1dAogICAgICAgICAgICBzID0gdGltZS50aW1lKCkKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgaW1nID0gcGlwZShwcm9tcHQ9aXRbInByb21wdCJdLCBuZWdhdGl2ZV9wcm9tcHQ9aXQuZ2V0KCJuZWdhdGl2ZSIsIG5lZyksIHdpZHRoPXcsIGhlaWdodD1oLAogICAgICAgICAgICAgICAgICAgICAgICAgICBudW1faW5mZXJlbmNlX3N0ZXBzPXN0ZXBzLCBndWlkYW5jZV9zY2FsZT02LjAsCiAgICAgICAgICAgICAgICAgICAgICAgICAgIGNhbGxiYWNrX29uX3N0ZXBfZW5kPXN0ZXBfZ3VhcmQoc3RlcHMsIFBBUkFNUy5nZXQoIml0ZW1fYnVkZ2V0X3MiLCAyNDApLCBpdFsiaWQiXSksCiAgICAgICAgICAgICAgICAgICAgICAgICAgIGdlbmVyYXRvcj10b3JjaC5HZW5lcmF0b3IoImN1ZGEiKS5tYW51YWxfc2VlZChpdC5nZXQoInNlZWQiLCAwKSArIGF0dGVtcHQpKS5pbWFnZXNbMF0KICAgICAgICAgICAgZXhjZXB0IChGbG9hdGluZ1BvaW50RXJyb3IsIFRpbWVvdXRFcnJvcikgYXMgZToKICAgICAgICAgICAgICAgIGZhaWxlZC5hcHBlbmQoeyJpZCI6IGl0WyJpZCJdLCAid2h5Ijogc3RyKGUpfSk7IGJyZWFrCiAgICAgICAgICAgIGlmIG5vdCBsb29rc19icm9rZW4oaW1nKToKICAgICAgICAgICAgICAgIGltZy5zYXZlKE9VVCAvIGYie2l0WydpZCddfS5wbmciKQogICAgICAgICAgICAgICAgZG9uZS5hcHBlbmQoeyJpZCI6IGl0WyJpZCJdLCAicyI6IHJvdW5kKHRpbWUudGltZSgpIC0gcywgMSksICJzaXplIjogZiJ7d314e2h9IiwgImF0dGVtcHQiOiBhdHRlbXB0ICsgMX0pCiAgICAgICAgICAgICAgICBicmVhawogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGZhaWxlZC5hcHBlbmQoeyJpZCI6IGl0WyJpZCJdLCAid2h5IjogImJsYWNrL05hTiBvdXRwdXQgdHdpY2UifSkKICAgICAgICByZXBvcnQoaW1hZ2VzPWRvbmUsIGZhaWxlZD1mYWlsZWQpCgoKZXJyID0gTm9uZQppZiBtb2RlbCA9PSAiemltYWdlIjoKICAgIHRyeToKICAgICAgICBnZW5femltYWdlKFBBUkFNU1siaXRlbXMiXSkKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZTogICAgICAgICAgICAgICAgICAgICAgICMgT09NIG9yIGJyb2tlbiBvdXRwdXQgb24gdGhpcyBHUFUg4oaSIGxpZ2h0IG1vZGVsLCBzYW1lIGl0ZW1zCiAgICAgICAgZXJyID0gZiJ7dHlwZShlKS5fX25hbWVfX306IHtzdHIoZSlbOjMwMF19IgogICAgaWYgZXJyOiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyBvdXRzaWRlIHRoZSBleGNlcHQgYmxvY2sgc28gdGhlIFotSW1hZ2UgcGlwZWxpbmUgaXMgZnJlZWQKICAgICAgICBpbXBvcnQgZ2M7IGdjLmNvbGxlY3QoKTsgdG9yY2guY3VkYS5lbXB0eV9jYWNoZSgpCiAgICAgICAgcmVwb3J0KHppbWFnZV9mYWlsZWQ9ZXJyLCBmYWxsYmFjaz0ic2R4bCIpCiAgICAgICAgUEFSQU1TWyJpdGVtcyJdID0gW2l0IGZvciBpdCBpbiBQQVJBTVNbIml0ZW1zIl0gaWYgbm90IChPVVQgLyBmIntpdFsnaWQnXX0ucG5nIikuZXhpc3RzKCldCiAgICAgICAgUEFSQU1TWyJtb2RlbCJdID0gInNkeGwiICAgICAgICAgICAgICAgICAjIHBlci1HUFUgY2hpbGRyZW4gcmUtcnVuIHRoaXMgZmlsZTogc2VuZCB0aGVtIHN0cmFpZ2h0IHRvIFNEWEwKICAgICAgICBnZW5fc2R4bChQQVJBTVNbIml0ZW1zIl0pCmVsc2U6CiAgICBnZW5fc2R4bChQQVJBTVNbIml0ZW1zIl0pCg==\'\nexec(base64.b64decode(_COMMON_B64).decode())\n\n\ndef _env_check():  # did Kaggle grant what the job asked for? (GPU + internet need a phone-verified account)\n    import shutil as _sh, socket as _so, subprocess as _sp\n    gpu = bool(_sh.which("nvidia-smi")) and _sp.run(["nvidia-smi", "-L"], capture_output=True).returncode == 0\n    try:\n        _so.getaddrinfo("huggingface.co", 443); net = True\n    except OSError:\n        net = False\n    report(gpu_granted=gpu, internet_granted=net)\n    if PARAMS.get("_need_gpu") and not gpu:\n        report(error=("Kaggle gave this session no GPU: this accelerator is busy/unavailable or another GPU session "\n                      "is running (free accounts get one at a time); retry later.") if net else\n                     ("Kaggle gave this session no GPU and no internet: the account needs phone verification "\n                      "(kaggle.com > Settings > Phone verification)."), gpu_unavailable_reason="busy" if net else "unverified")\n        raise SystemExit(3)\n    if PARAMS.get("_need_internet") and not net:\n        report(error="Kaggle gave this session no internet (phone verification needed, or internet off).")\n        raise SystemExit(3)\n\n\n_env_check()\ntry:\n    import torch as _t\n    if _t.cuda.is_available():\n        report(gpu=gpu_info())\nexcept Exception:\n    pass\ntry:\n    exec(compile(base64.b64decode(_JOB_B64).decode(), "job.py", "exec"))\nexcept _Done:\n    pass\nif json.loads((OUT / "report.json").read_text()).get("error"):\n    raise SystemExit(4)\n'
open('/content/vs_job.py', 'w').write(JOB)
import subprocess, sys
rc = subprocess.run([sys.executable, '/content/vs_job.py']).returncode
print('job exit code', rc, '(0 = ok; see report.json in the zip either way)')


In [ ]:
# 4. pack the results and download them
import json, pathlib, shutil
out = pathlib.Path('/kaggle/working/out')
rep = out / 'report.json'
if rep.exists():
    d = json.loads(rep.read_text()); d['provider'] = 'colab'; rep.write_text(json.dumps(d, indent=2))
    print(json.dumps(d, indent=1)[:3000])
zp = shutil.make_archive('/content/vs-result-sdxl', 'zip', out)
from google.colab import files
files.download(zp)


In [ ]:
# 5. release the GPU (run after the download has started)
from google.colab import runtime
runtime.unassign()
